# MADE for the 1D lattice phi^4 model - fast sampling

Same model and training idea as `madesym.ipynb` (autoregressive network, a mixture of two Gaussians per site, reverse-KL training against `exp(-E)`),
with **sampling that is about 90x cheaper**.

## What was slow

`NN_Sample` generated one configuration site by site, and for **every** site it called the whole network on the whole batch:

```python
for cell in range(100):
    params[:, cell, :] = MADE_V(configs)[:, cell, :]      # full network, only 5 of its 500 outputs are used
```

* One call costs `N x D x H` (first layer) + `N x H x (D*5)` (output layer) multiply-adds, about 2.5 GFLOP for `N = 1024, D = 100, H = 2048`; it is done
  100 times, i.e. **~250 GFLOP per batch, of which 99% is thrown away** (the other 99 sites' outputs, and hidden contributions of sites not sampled yet).
* The loop ran in eager mode, copied the parameters device -> numpy every site, and built a `tfd.Mixture` / `tfd.Categorical` object 100 times per batch.
* The training step then evaluated the network two more times (`KL_Divergence`, `KL_loss`) on top of that.

## The fix

For a masked network, site `i` depends only on `x_0 ... x_{i-1}`, and the hidden pre-activation is a *sum over inputs*:
`h_pre = b1 + sum_j 0.5 x_j W1[j, :]`. So

1. keep `h_pre` for the whole batch; after sampling site `i` add **one row** (`0.5 x_i W1[i, :]`): cost `N x H`;
2. compute only site `i`'s 5 outputs from `relu(h_pre)`: cost `N x H x 5`;
3. sample the two-Gaussian mixture by hand (no distribution objects), and run the whole loop inside one `tf.function` (`tf.while_loop`).

This is ~2.7 GFLOP per batch (about 90x less) and identical in distribution: given the same random numbers it produces the same configurations
(tested in NumPy: max difference 6e-6, float32 rounding; the cells below re-test it in TensorFlow). The 100 steps remain sequential (that is inherent to autoregressive sampling),
but each is now tiny.

The network is written out explicitly (masked weights, same architecture as before: `x/2 -> 2048 relu -> 5 params per site`) instead of using `tfb.AutoregressiveNetwork`,
because the fast loop needs direct access to the weights. **Checkpoints of the old TFP layer do not load into this one.**

In [ ]:
import time
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp          # only used for the mixture log-probability below
tfd = tfp.distributions
from tensorflow import keras
from tensorflow.keras.utils import Progbar
DTYPE = tf.float32

In [ ]:
class MADE(keras.layers.Layer):
    """One-hidden-layer MADE: x [N, D] -> [N, D, 5] parameters (loc1, loc2, scale1, scale2, alpha) of a two-Gaussian mixture per site.
       Site i depends only on x[:, :i].   call()   evaluates all sites at once (training, log-probability);
                                           sample() draws the sites one by one, but reuses the hidden pre-activation (cheap)."""
    def __init__(self, input_dim, hidden=2048, n_params=5, name='MADE'):
        super().__init__(name=name)
        D, H, P = input_dim, hidden, n_params
        self.D, self.H, self.P = D, H, P
        m = 1 + np.arange(H) % (D - 1)                   # hidden degrees 1..D-1 (input j has degree j+1, output site i has degree i+1)
        self.mask1 = tf.constant((m[None, :] >= np.arange(1, D + 1)[:, None]).astype(np.float32))                                   # [D, H]
        self.mask2 = tf.constant(np.broadcast_to(np.arange(1, D + 1)[None, :, None] > m[:, None, None], (H, D, P)).astype(np.float32))  # [H, D, P]
        self.W1 = self.add_weight(name='W1', shape=(D, H), initializer='glorot_uniform')
        self.b1 = self.add_weight(name='b1', shape=(H,), initializer='zeros')
        self.W2 = self.add_weight(name='W2', shape=(H, D * P), initializer='glorot_uniform')
        self.b2 = self.add_weight(name='b2', shape=(D, P), initializer='zeros')

    def _post(self, o):
        """raw outputs [..., 5] -> (loc1, loc2, scale1, scale2, alpha); same transformations as the original NN layer"""
        l1, l2, s1, s2, a = tf.unstack(o, axis=-1)
        return tf.stack([4 * tf.nn.tanh(l1) + 0.4, 4 * tf.nn.tanh(l2) - 0.4, tf.nn.sigmoid(s1), tf.nn.sigmoid(s2), tf.nn.sigmoid(a)], axis=-1)

    def _W2(self):
        return tf.reshape(self.W2, [self.H, self.D, self.P]) * self.mask2

    def call(self, x):                                   # [N, D] -> [N, D, P]
        h = tf.nn.relu(tf.matmul(0.5 * x, self.W1 * self.mask1) + self.b1)
        return self._post(tf.einsum('nh,hdp->ndp', h, self._W2()) + self.b2)

    def _run(self, n, x_given=None):
        """Site-by-site loop with the incremental hidden pre-activation. x_given=None: sample each site from its mixture.
           x_given=[n, D]: teacher forcing (used only to test the loop against call()). Returns (x [n, D], params [n, D, P])."""
        D = self.D
        W1, W2 = self.W1 * self.mask1, self._W2()
        h_pre = tf.zeros([n, self.H]) + self.b1          # b1 + contributions of the sites already sampled
        xs, ps = tf.TensorArray(tf.float32, size=D), tf.TensorArray(tf.float32, size=D)
        def body(i, h_pre, xs, ps):
            par = self._post(tf.matmul(tf.nn.relu(h_pre), tf.gather(W2, i, axis=1)) + tf.gather(self.b2, i))   # only site i: [n, P]
            if x_given is None:
                l1, l2, s1, s2, a = tf.unstack(par, axis=-1)
                first = tf.random.uniform([n]) < a                                  # component 1 with probability alpha
                x = tf.where(first, l1, l2) + tf.where(first, s1, s2) * tf.random.normal([n])
            else:
                x = x_given[:, i]
            h_pre = h_pre + 0.5 * x[:, None] * tf.gather(W1, i, axis=0)[None, :]     # add the contribution of site i: one row of W1
            return i + 1, h_pre, xs.write(i, x), ps.write(i, par)
        _, _, xs, ps = tf.while_loop(lambda i, *_: i < D, body, [tf.constant(0), h_pre, xs, ps])
        return tf.transpose(xs.stack()), tf.transpose(ps.stack(), [1, 0, 2])

    @tf.function
    def sample(self, n):
        return self._run(n)[0]

    @tf.function
    def params_teacher_forced(self, x):
        return self._run(tf.shape(x)[0], x)[1]

input_dim = 100
MADE_V = MADE(input_dim)
MADE_V.build((None, input_dim))
print('parameters:', sum(int(np.prod(v.shape)) for v in MADE_V.trainable_variables))

## Tests: autoregressive property, fast loop == full pass, and timing

In [ ]:
# 1) site i must not depend on x_j for j >= i   (strictly autoregressive)
x_t = tf.random.normal([4, input_dim]) * 1.5
with tf.GradientTape() as tape:
    tape.watch(x_t); out_t = MADE_V(x_t)
jac = tape.batch_jacobian(out_t, x_t)                                     # [N, site i, param, input j]
not_allowed = (np.arange(input_dim)[None, :] >= np.arange(input_dim)[:, None]).astype(np.float32)     # [i, j], j >= i
print('max |d out_i / d x_j| for j >= i (must be 0):', float(tf.reduce_max(tf.abs(jac) * not_allowed[None, :, None, :])))

# 2) the incremental loop gives the same conditional parameters as the full pass
print('incremental loop vs full pass, max abs diff    :', float(tf.reduce_max(tf.abs(MADE_V(x_t) - MADE_V.params_teacher_forced(x_t)))))

# 3) the parameters used while sampling are those of the full pass evaluated on the samples (so sampling == the density used in the loss)
xs_t, ps_t = MADE_V._run(256)
print('params used in sampling vs full pass on samples :', float(tf.reduce_max(tf.abs(ps_t - MADE_V(xs_t)))))

In [ ]:
def NN_Sample_slow(sample_size):
    """the ORIGINAL scheme (full network at every site, eager, numpy copies, tfd.Mixture objects) - for the timing comparison only"""
    configs = np.zeros((sample_size, input_dim), dtype=np.float32)
    for cell in np.arange(input_dim):
        params = MADE_V(configs)[:, cell, :].numpy()
        shifts_1, shifts_2, scales_1, scales_2, mix = [params[:, k] for k in range(5)]
        bimix = tfd.Mixture(cat=tfd.Categorical(probs=tf.stack([mix, 1. - mix], axis=-1)),
                            components=[tfd.Normal(loc=shifts_1, scale=scales_1), tfd.Normal(loc=shifts_2, scale=scales_2)])
        configs[:, cell] = bimix.sample()
    return configs

n_t = 256
MADE_V.sample(n_t)                                                        # trace / warm up
t0 = time.time(); a = NN_Sample_slow(n_t); t_slow = time.time() - t0
t0 = time.time(); b = MADE_V.sample(n_t).numpy(); t_fast = time.time() - t0
print(f'n = {n_t}: original sampler {t_slow:.2f} s, new sampler {t_fast:.3f} s  ->  x{t_slow / t_fast:.0f} faster')
t0 = time.time(); MADE_V.sample(1024).numpy(); print(f'new sampler, n = 1024 (the training batch): {time.time() - t0:.3f} s')
print('per-site mean, |old - new| averaged over sites (statistical noise only):', float(np.abs(a.mean(0) - b.mean(0)).mean()),
      '  per-site std:', float(np.abs(a.std(0) - b.std(0)).mean()))

## Energy, log-probability and loss (unchanged)

In [ ]:
# the lattice phi^4 energy (periodic chain)
@tf.function
def Energy(y):
    lamd = 1.
    f = np.sqrt(2.)
    a = 0.1
    m = 0.5
    kinetic_energy = tf.reduce_sum(m / (2 * a) * (y - tf.roll(y, shift=-1, axis=1))**2, axis=-1)
    potential_energy = tf.reduce_sum(lamd * (y**2 - f**2)**2, axis=-1)
    return kinetic_energy + a * potential_energy

@tf.function
def log_probability(xs, output):
    shift_1, shift_2, scale_1, scale_2, mix = tf.unstack(output, 5, axis=-1)
    bimix_gauss = tfd.Mixture(
        cat=tfd.Categorical(probs=tf.stack([mix, 1 - mix], axis=-1)),
        components=[tfd.Normal(loc=shift_1, scale=scale_1), tfd.Normal(loc=shift_2, scale=scale_2)])
    return tf.reduce_sum(bimix_gauss.log_prob(xs), axis=-1)

## Training

Same objective as before, `loss = mean( log_p (E + 0.5 log_p) / |KL| )` with `KL = mean(log_p + E)` treated as a constant, but the network is evaluated **once** per step
(before: twice, on top of the sampling loop) and the whole step is one `tf.function`.

In [ ]:
learning_rate = 1e-5
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
checkpoint_directory = './saved_models_MADE/'
checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=MADE_V)
latest = tf.train.latest_checkpoint(checkpoint_directory)
if latest:
    checkpoint.restore(latest)
    print('restored', latest)

@tf.function
def train_step(n_samples):
    xs = MADE_V.sample(n_samples)                      # fast autoregressive sampling (outside the tape: samples are constants)
    es = Energy(xs)
    with tf.GradientTape() as tape:
        log_p = log_probability(xs, MADE_V(xs))
        KL_D = tf.stop_gradient(tf.reduce_mean(log_p + es))
        loss = tf.reduce_mean(log_p * (es + 0.5 * log_p) / tf.abs(KL_D))
    grads = tape.gradient(loss, MADE_V.trainable_variables)
    optimizer.apply_gradients(zip(grads, MADE_V.trainable_variables))
    return KL_D

n_epochs, n_iter, n_samples = 20, 1000, 1024
for epoch in range(n_epochs):
    print('Epoch {:}/{:}'.format(epoch, n_epochs))
    progbar = Progbar(n_iter)
    losses = []
    for it in range(n_iter):
        kl = float(train_step(n_samples))
        losses.append(kl)
        progbar.add(1, values=[('loss', kl)])
    print('  Mean loss: {}'.format(np.mean(losses)))
    print('  Var of loss: {}'.format(np.var(losses)))
    checkpoint.save(checkpoint_directory + 'ckpt_{}'.format(epoch))